# RAG Pipeline Setup on AMD Droplet

Complete guide to:
1. Clone the repo
2. Set up Python environment
3. Install dependencies
4. Build search index
5. Start FastAPI + Frontend

Run each cell in order.

## Step 1: Install System Dependencies

Run this on the droplet terminal first (one-time only):

In [ ]:
# These commands should be run in terminal, not in Jupyter
# SSH into your droplet and run:
"""
apt update && apt install -y \
    git python3.11-venv python3-dev \
    nodejs npm build-essential libssl-dev
"""

## Step 2: Clone the Repository

In [ ]:
import os
import subprocess

# Change to home directory
os.chdir(os.path.expanduser('~'))

# Clone the repo (replace with your repo URL)
result = subprocess.run(
    ['git', 'clone', 'https://github.com/your-username/rag_pipeline.git'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("✓ Repository cloned successfully!")
    
# Navigate to project
os.chdir('rag_pipeline')
print(f"\nCurrent directory: {os.getcwd()}")

## Step 3: Create Python Virtual Environment

In [ ]:
import os
import subprocess

# Create virtual environment
result = subprocess.run(
    ['python3', '-m', 'venv', 'venv'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Virtual environment created!")
else:
    print(f"Error: {result.stderr}")

# Verify
venv_path = os.path.join(os.getcwd(), 'venv', 'bin', 'python')
print(f"\nVenv Python: {venv_path}")
print(f"Exists: {os.path.exists(venv_path)}")

## Step 4: Install Python Dependencies

In [ ]:
import subprocess
import os

# Get path to venv python
venv_python = os.path.join(os.getcwd(), 'venv', 'bin', 'python')
venv_pip = os.path.join(os.getcwd(), 'venv', 'bin', 'pip')

# Upgrade pip
print("Upgrading pip...")
result = subprocess.run(
    [venv_pip, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ pip upgraded")

# Install requirements
print("\nInstalling dependencies from requirements.txt...")
result = subprocess.run(
    [venv_pip, 'install', '-r', 'requirements.txt'],
    capture_output=True,
    text=True,
    timeout=300
)

if result.returncode == 0:
    print("✓ All dependencies installed!")
    print("\nOutput (last 20 lines):")
    print('\n'.join(result.stdout.split('\n')[-20:]))
else:
    print(f"Error: {result.stderr}")

## Step 5: Verify PyTorch on AMD ROCm

In [ ]:
import subprocess
import os

venv_python = os.path.join(os.getcwd(), 'venv', 'bin', 'python')

# Check PyTorch
check_code = """
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
print(f'Device Count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'HIP Version: {torch.version.hip}')
"""

result = subprocess.run(
    [venv_python, '-c', check_code],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

## Step 6: Add Policy Documents

Place your documents in `data/docs/` before building the index.

In [ ]:
import os

# Create docs directory
docs_dir = os.path.join(os.getcwd(), 'data', 'docs')
os.makedirs(docs_dir, exist_ok=True)

print(f"Docs directory: {docs_dir}")
print(f"Exists: {os.path.exists(docs_dir)}")
print("\nSupported file types: .pdf, .docx, .doc, .txt, .md")
print("\nCurrently in docs/:")

if os.listdir(docs_dir):
    for f in os.listdir(docs_dir):
        print(f"  - {f}")
else:
    print("  (empty - add documents here)")

## Step 7: Build FAISS Index

⚠️ **Warning:** This downloads ~430 MB of models and may take 5-10 minutes.

In [ ]:
import subprocess
import os

venv_python = os.path.join(os.getcwd(), 'venv', 'bin', 'python')

print("Building FAISS index...")
print("This may take 5-10 minutes.\n")

result = subprocess.run(
    [venv_python, 'scripts/build_index.py'],
    capture_output=True,
    text=True,
    timeout=600
)

print(result.stdout)

if result.returncode == 0:
    print("\n✓ Index built successfully!")
    
    # Check files
    index_dir = os.path.join(os.getcwd(), 'data', 'index')
    print(f"\nFiles in {index_dir}:")
    for f in os.listdir(index_dir):
        path = os.path.join(index_dir, f)
        size = os.path.getsize(path) / (1024*1024)  # MB
        print(f"  - {f} ({size:.2f} MB)")
else:
    print(f"Error: {result.stderr}")

## Step 8: Start FastAPI + Frontend

Run the startup script. This starts both services.

In [ ]:
import subprocess
import os

print("Starting FastAPI + Frontend...\n")
print("This will start two services:")
print("  - Backend: http://your_droplet_ip:8000")
print("  - Frontend: http://your_droplet_ip:5173")
print("  - API Docs: http://your_droplet_ip:8000/docs\n")

# Check if using setup_and_run.sh (Linux)
if os.path.exists('setup_and_run.sh'):
    print("Running bash setup_and_run.sh...\n")
    # Note: In Jupyter, this would need to be run in terminal
    print("Run this in a terminal:")
    print("  bash setup_and_run.sh")
else:
    print("setup_and_run.sh not found")

## Summary

You've successfully:
1. ✓ Cloned the repository
2. ✓ Created Python virtual environment
3. ✓ Installed all dependencies
4. ✓ Built FAISS search index
5. ✓ Ready to start services

### Next Steps

**Open a terminal and run:**
```bash
cd ~/rag_pipeline
source venv/bin/activate
bash setup_and_run.sh
```

Or run in background:
```bash
bash setup_and_run.sh &
```

### Access Your Services

- **Backend API**: `http://your_droplet_ip:8000`
- **API Documentation**: `http://your_droplet_ip:8000/docs` (interactive)
- **Frontend**: `http://your_droplet_ip:5173` (or configured port)

### Systemd Service (Optional Production Setup)

```bash
sudo cp rag-api.service /etc/systemd/system/
sudo systemctl daemon-reload
sudo systemctl start rag-api
sudo systemctl enable rag-api
```

### Troubleshooting

| Issue | Fix |
|-------|-----|
| "npm not found" | Run in terminal: `sudo apt install nodejs npm` |
| Port already in use | Change `API_PORT` or `FRONTEND_PORT` in `.env` |
| FAISS build fails | Make sure you have at least 4 GB free RAM |
| GPU not detected | Check ROCm: `rocm-smi` |

Enjoy your RAG pipeline! 🚀